In [5]:
import pandas as pd
import glob

In [7]:
# Data Quality Validation
files = glob.glob("data/raw/games/games_*.csv")

results = []

# For each data file of games, we will check various data points for the file
for file in files:

    games = pd.read_csv(file)

    fbs_games = games[
        (games["homeClassification"] == "fbs") | (games["awayClassification"] == "fbs")
    ].copy()

    results.append({
        "season": games["season"].iloc[0],
        "all_games": len(games),
        "fbs_games": len(fbs_games),
        "fbs_vs_fbs": (
            (fbs_games["homeClassification"] == "fbs") & (fbs_games["awayClassification"] == "fbs")
        ).sum(),
        "fbs_vs_fcs": (
            (
                (fbs_games["homeClassification"] == "fbs") & (fbs_games["awayClassification"] == "fcs")
            ) |
            (
                (fbs_games["homeClassification"] == "fcs") & (fbs_games["awayClassification"] == "fbs")
            )
        ).sum(),
        "missing_home_class": games["homeClassification"].isna().sum(),
        "missing_away_class": games["awayClassification"].isna().sum(),
        "duplicate_ids": games["id"].duplicated().sum(),
        "missing_home_score": games["homePoints"].isna().sum(),
        "missing_away_score": games["awayPoints"].isna().sum()
    })

# Report the results
audit = pd.DataFrame(results).sort_values("season")

print(audit.to_string(index = False))

 season  all_games  fbs_games  fbs_vs_fbs  fbs_vs_fcs  missing_home_class  missing_away_class  duplicate_ids  missing_home_score  missing_away_score
   2015       1491        829         724         105                   0                  12              0                   0                   0
   2016       1502        832         719         113                   1                  11              0                   0                   0
   2017       1505        834         736          98                   1                  10              0                   0                   0
   2018       1511        845         733         112                   1                  14              0                   0                   0
   2019       1577        848         734         114                   0                  23              0                   0                   0
   2020        563        542         508          34                   0                   0             

In [9]:
# The quality check showed 8 games with missing scores for one or both teams. Investigate those games
files = glob.glob("data/raw/games/games_*.csv")

for file in files:

    games = pd.read_csv(file)

    missing_scores = games[
        games["homePoints"].isna() |
        games["awayPoints"].isna()
    ]

    if len(missing_scores) > 0:

        print(f"\n{'=' * 60}")
        print(f"SEASON: {games['season'].iloc[0]}")
        print(f"{'=' * 60}")

        print(
            missing_scores[
                [
                    "id",
                    "season",
                    "week",
                    "seasonType",
                    "completed",
                    "startDate",
                    "homeTeam",
                    "homeClassification",
                    "awayTeam",
                    "awayClassification",
                    "homePoints",
                    "awayPoints",
                    "notes"
                ]
            ].to_string(index=False)
        )


SEASON: 2023
       id  season  week seasonType  completed                startDate        homeTeam homeClassification         awayTeam awayClassification  homePoints  awayPoints notes
401552878    2023     9    regular      False 2023-10-28T17:00:00.000Z           Colby                iii       Middlebury                iii         NaN         NaN   NaN
401550299    2023     9    regular      False 2023-10-28T17:00:00.000Z           Bates                iii         Williams                iii         NaN         NaN   NaN
401549719    2023     9    regular      False 2023-10-28T17:00:00.000Z         Bowdoin                iii     Trinity (CT)                iii         NaN         NaN   NaN
401552884    2023    11    regular      False 2023-11-12T17:00:00.000Z Worcester State                iii Framingham State                iii         NaN         NaN   NaN

SEASON: 2024
       id  season  week seasonType  completed                startDate  homeTeam homeClassification         away

In [ ]:
# After this data check, the master data file for games will follow the following 3 guidlines:
# 1. Keep only games involving one or more FBS teams
# 2. Keep only completed games
# 3. Require a final score for both the home and away team in order to pass through to the final dataset

In [13]:
# Moving on to stats data, checking information about the source prior to cleaning
df = pd.read_csv("data/raw/stats/team_stats_2025.csv")

print(df.shape)
print(df.head())

print("\nUnique teams:", df["team"].nunique())
print("\nUnique statistics:", df["statName"].nunique())

print("\nStatistics:")
print(df["statName"].unique())

print("\nMissing values:")
print(df.isna().sum())

print("\nRecords per team:")
print(df.groupby("team").size().describe())

(8568, 5)
   season       team     conference                       statName  statValue
0    2025  Air Force  Mountain West                     firstDowns        264
1    2025  Air Force  Mountain West             firstDownsOpponent        247
2    2025  Air Force  Mountain West          fourthDownConversions         24
3    2025  Air Force  Mountain West  fourthDownConversionsOpponent         10
4    2025  Air Force  Mountain West                    fourthDowns         34

Unique teams: 136

Unique statistics: 63

Statistics:
<StringArray>
[                   'firstDowns',            'firstDownsOpponent',
         'fourthDownConversions', 'fourthDownConversionsOpponent',
                   'fourthDowns',           'fourthDownsOpponent',
                   'fumblesLost',           'fumblesLostOpponent',
              'fumblesRecovered',      'fumblesRecoveredOpponent',
                         'games',                 'interceptions',
         'interceptionsOpponent',               'in

In [15]:
# Inspect game-level stats
game_stats = pd.read_csv(
    "data/raw/game_stats/game_stats_2025.csv"
)

print(game_stats.shape)
print(game_stats.columns.tolist())
print(game_stats.head())
print(game_stats.dtypes)

print(game_stats.isna().sum())

print(game_stats["gameId"].nunique())

(3216, 8)
['gameId', 'season', 'seasonType', 'week', 'team', 'opponent', 'offense', 'defense']
      gameId  season seasonType  week           team       opponent  \
0  401752665    2025    regular     1        Alabama  Florida State   
1  401752665    2025    regular     1  Florida State        Alabama   
2  401752666    2025    regular     1    Alabama A&M       Arkansas   
3  401752666    2025    regular     1       Arkansas    Alabama A&M   
4  401752667    2025    regular     1         Auburn         Baylor   

                                             offense  \
0  {'plays': 72, 'drives': 10, 'ppa': 0.100224770...   
1  {'plays': 63, 'drives': 10, 'ppa': 0.290056628...   
2  {'plays': 55, 'drives': 12, 'ppa': 0.050584981...   
3  {'plays': 75, 'drives': 13, 'ppa': 0.407195495...   
4  {'plays': 70, 'drives': 9, 'ppa': 0.2888159132...   

                                             defense  
0  {'plays': 63, 'drives': 10, 'ppa': 0.290056628...  
1  {'plays': 72, 'drives': 10, 

In [16]:
import pandas as pd
import ast

game_stats = pd.read_csv(
    "data/raw/game_stats/game_stats_2025.csv"
)

offense = ast.literal_eval(game_stats.loc[0, "offense"])
defense = ast.literal_eval(game_stats.loc[0, "defense"])

print("OFFENSE:")
print(offense.keys())

print("\nDEFENSE:")
print(defense.keys())

print("\nOFFENSE VALUES:")
print(offense)

print("\nDEFENSE VALUES:")
print(defense)

OFFENSE:
dict_keys(['plays', 'drives', 'ppa', 'totalPPA', 'successRate', 'explosiveness', 'powerSuccess', 'stuffRate', 'lineYards', 'lineYardsTotal', 'secondLevelYards', 'secondLevelYardsTotal', 'openFieldYards', 'openFieldYardsTotal', 'standardDowns', 'passingDowns', 'rushingPlays', 'passingPlays'])

DEFENSE:
dict_keys(['plays', 'drives', 'ppa', 'totalPPA', 'successRate', 'explosiveness', 'powerSuccess', 'stuffRate', 'lineYards', 'lineYardsTotal', 'secondLevelYards', 'secondLevelYardsTotal', 'openFieldYards', 'openFieldYardsTotal', 'standardDowns', 'passingDowns', 'rushingPlays', 'passingPlays'])

OFFENSE VALUES:
{'plays': 72, 'drives': 10, 'ppa': 0.10022477071900321, 'totalPPA': 7.216183491768231, 'successRate': 0.375, 'explosiveness': 1.366452788630703, 'powerSuccess': 0.6666666666666666, 'stuffRate': 0.18518518518518517, 'lineYards': 3.3296296296296295, 'lineYardsTotal': 90, 'secondLevelYards': 0.8888888888888888, 'secondLevelYardsTotal': 24, 'openFieldYards': 0.2222222222222222, '

In [20]:
# Check week 0 games
games_2025 = pd.read_csv("data/raw/games/games_2025.csv")

stats_2025 = pd.read_csv("data/raw/game_team_stats/game_team_stats_2025.csv")

print(games_2025.shape)
print(stats_2025.shape)

print(stats_2025["gameId"].isin(games_2025["id"]).value_counts())

print(games_2025[
    ["id", "season", "week", "startDate", "homeTeam", "awayTeam"]
].sort_values("startDate").head(15))

print(games_2025[
    (games_2025["homeTeam"] == "Kansas State") |
    (games_2025["awayTeam"] == "Kansas State")
][["id", "week", "startDate", "homeTeam", "awayTeam"]].sort_values("startDate"))

(3745, 34)
(3250, 43)
gameId
True    3250
Name: count, dtype: int64
           id  season  week                 startDate              homeTeam  \
0   401756846    2025     1  2025-08-23T16:00:00.000Z          Kansas State   
1   401767476    2025     1  2025-08-23T17:00:00.000Z              Nicholls   
2   401760371    2025     1  2025-08-23T20:00:00.000Z                  UNLV   
3   401767126    2025     1  2025-08-23T20:30:00.000Z        Portland State   
4   401756847    2025     1  2025-08-23T22:30:00.000Z                Kansas   
5   401757218    2025     1  2025-08-23T23:00:00.000Z      Western Kentucky   
6   401754516    2025     1  2025-08-23T23:30:00.000Z               Hawai'i   
7   401767410    2025     1  2025-08-23T23:30:00.000Z              Southern   
8   401773590    2025     1  2025-08-28T20:00:00.000Z    Central Washington   
9   401762522    2025     1  2025-08-28T21:30:00.000Z         South Florida   
19  401773703    2025     1  2025-08-28T22:00:00.000Z          

In [26]:
import pandas as pd

master_games = pd.read_csv("data/master/master_game_data.csv")

master_cols = set(master_games.columns)

for year in range(2015, 2026):
    path = f"data/processed/game_team_stats/game_team_stats_{year}.csv"
    games = pd.read_csv(path)

    year_cols = set(games.columns)

    only_in_year = sorted(year_cols - master_cols)
    only_in_master = sorted(master_cols - year_cols)

    print(f"\n{'=' * 60}")
    print(f"{year}")
    print(f"{'=' * 60}")

    print(f"Year columns:    {len(year_cols)}")
    print(f"Master columns:  {len(master_cols)}")

    if only_in_year:
        print(f"\nColumns in {year} but NOT master ({len(only_in_year)}):")
        for col in only_in_year:
            print(f"  {col}")

    if only_in_master:
        print(f"\nColumns in master but NOT {year} ({len(only_in_master)}):")
        for col in only_in_master:
            print(f"  {col}")

    if not only_in_year and not only_in_master:
        print("\n✓ Column sets match")


2015
Year columns:    279
Master columns:  279

✓ Column sets match

2016
Year columns:    279
Master columns:  279

✓ Column sets match

2017
Year columns:    279
Master columns:  279

✓ Column sets match

2018
Year columns:    279
Master columns:  279

✓ Column sets match

2019
Year columns:    279
Master columns:  279

✓ Column sets match

2020
Year columns:    279
Master columns:  279

✓ Column sets match

2021
Year columns:    279
Master columns:  279

✓ Column sets match

2022
Year columns:    279
Master columns:  279

✓ Column sets match

2023
Year columns:    279
Master columns:  279

✓ Column sets match

2024
Year columns:    279
Master columns:  279

✓ Column sets match

2025
Year columns:    279
Master columns:  279

✓ Column sets match


In [27]:
print("Master shape:", master_games.shape)

print("\nRows by season:")
print(master_games["season"].value_counts().sort_index())

print("\nUnique games by season:")
print(
    master_games.groupby("season")["gameId"]
    .nunique()
    .sort_index()
)

print("\nDuplicate game/team rows:")
print(
    master_games.duplicated(subset=["gameId", "team"]).sum()
)

Master shape: (23622, 279)

Rows by season:
season
2015    1658
2016    1662
2017    1668
2018    1690
2019    1696
2020    1046
2021    1698
2022    3064
2023    2980
2024    3210
2025    3250
Name: count, dtype: int64

Unique games by season:
season
2015     829
2016     831
2017     834
2018     845
2019     848
2020     523
2021     849
2022    1532
2023    1490
2024    1605
2025    1625
Name: gameId, dtype: int64

Duplicate game/team rows:
0
